In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

# In WebAssembly / Pyodide / JupyterLite environments, pre-load dependencies
if 'pyodide' in sys.modules or hasattr(sys, '_emscripten_info') or sys.platform == 'emscripten':
    try:
        import piplite
        await piplite.install(['pyyaml', 'matplotlib', 'ipywidgets', 'numpy'])
    except Exception:
        pass

# Explicit imports for Pyodide AST scanner and interactive widget ecosystem
try:
    import yaml
except ImportError:
    pass
import matplotlib.pyplot as plt
import ipywidgets as widgets

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_12_monte_carlo_foundations.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 12 — Monte Carlo Foundations, Curse of Dimensionality & Non-Uniform Sampling

_Session 506a4b0c · Estimated time: ~115 minutes · 38 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Explain the historical origin and core philosophy of Monte Carlo methods (Ulam, von Neumann, and Metropolis at Los Alamos in the 1940s; Buffon's needle; stochastic vs. deterministic evaluation of integrals).
- Derive and apply hit-or-miss Monte Carlo estimation of π in the unit square, including its binomial hit probability p = π/4 and statistical standard error scaling σ ∝ M^(-1/2).
- Prove the Curse of Dimensionality scaling law for order-α deterministic quadrature in D dimensions (CPU time ∝ ε^(-D/α)) versus Monte Carlo integration (CPU time ∝ ε^(-2)), and identify the exact crossover dimension D* = 2α.
- Compute moments, expectations, cumulative distribution functions (CDFs), and apply the fundamental Jacobian transformation law p_y(y)|dy| = p_x(x)|dx| for 1D and multivariate random variable mappings.
- Derive and execute Inverse Transform Sampling x = C^(-1)(u) for exponential/Poisson, power-law, and Lorentzian/Cauchy distributions, as well as the 2D Box–Muller and Marsaglia polar transforms for Normal Gaussian deviates.


In [2]:
ws.render_section('sec_mc_intro_and_pi')

### Origins and Philosophy of Monte Carlo
In the 1940s at Los Alamos National Laboratory, **Stanislaw Ulam**, **John von Neumann**, and **Nicholas Metropolis** coined the name **Monte Carlo**—after the famous casino in Monaco where Ulam's uncle borrowed money to gamble—to describe **any numerical algorithm that uses (pseudo)random numbers as an essential part of the calculation**.

Unlike deterministic Molecular Dynamics (which integrates Newton's equations of motion along a time-ordered trajectory), Monte Carlo (MC) is fundamentally **stochastic**:
- It evaluates **high-dimensional integrals** (such as the canonical configurational partition function $Z = \int d\mathbf{r}_1 \cdots d\mathbf{r}_N\, e^{-\beta V(\mathbf{r}^N)}$ in $D = 3N$ dimensions) by randomly sampling points in the integration domain.
- In statistical physics, it is most often implemented as a **Markov chain** known as **Metropolis Monte Carlo**.
- Its earliest forerunner is **Buffon's Needle** (1777), which estimated $\pi$ from the probability that a dropped needle crosses parallel floorboard lines.

### Simple Monte Carlo: Throwing Darts to Estimate $\pi$
As shown on Slide 2, consider a circle of radius $r = 1$ inscribed inside a square, and focus on **one quadrant** in the unit square $[0, 1] \times [0, 1]$:
$$\frac{\text{Area of } 1/4\text{ circle}}{\text{Area of unit square}} = \frac{\pi r^2 / 4}{r^2} = \frac{\pi}{4} \approx 0.785398$$

If we throw $M$ independent random darts $(x_n, y_n)$ with $x_n \sim \text{Uniform}(0, 1)$ and $y_n \sim \text{Uniform}(0, 1)$:
```python
hits = 0
for n in range(1, M + 1):
    x = rng.random()
    y = rng.random()
    if (x**2 + y**2 <= 1.0):
        hits += 1
pi_est = 4.0 * hits / M
```
Each dart is an independent Bernoulli trial with success probability $p = \frac{\pi}{4}$ and variance per trial $\sigma_1^2 = p(1 - p) = \frac{\pi}{4}\left(1 - \frac{\pi}{4}\right)$. After $M$ throws:
- Expected fraction of hits: $\langle \text{hits}/M \rangle = p = \frac{\pi}{4} \implies \langle \hat{\pi} \rangle = 4p = \pi$.
- Variance of the hit fraction: $\text{Var}(\hat{p}) = \frac{p(1-p)}{M}$.
- **Standard error of $\hat{\pi} = 4\hat{p}$:**
  $$\sigma_{\hat{\pi}} = 4\sqrt{\frac{p(1-p)}{M}} = \sqrt{\frac{\pi(4 - \pi)}{M}} \approx \frac{1.642}{\sqrt{M}}$$


In [3]:
ws.render_section('concept_hit_or_miss_pi')

### Interactive Explorer: Hit-or-Miss Dartboard & M^(-1/2) Convergence of π

Throw $M$ random darts into the unit square $[0, 1]^2$ (Slide 2). Switch between the **2D Dartboard View** and the **Running Convergence View** to see how $\hat{\pi}(M) = 4 N_{\text{hit}}/M$ converges inside the theoretical $\pm 1.96\sqrt{\pi(4-\pi)/M}$ confidence band.


**Key Takeaway:** Because the Monte Carlo statistical error decays as $\varepsilon \propto M^{-1/2}$, reducing the error by a factor of $10$ (gaining **one extra decimal place** of precision) requires **$10^2 = 100$ times more samples**!


In [4]:
ws.render_section('q01_mc_definition_history')

**Q1 · easy** · 💡

What is the defining characteristic of a **Monte Carlo method** (named at Los Alamos in the 1940s), as contrasted with deterministic grid or trajectory methods?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check Slide 1: what is an essential ingredient of every Monte Carlo algorithm?

</div>
</details>

- [ ] **(A)** It is a stochastic algorithm that uses (pseudo)random numbers as an essential part of evaluating integrals or sampling probability distributions.
- [ ] **(B)** It is a deterministic finite-difference solver for Newton's second law $F = ma$.
- [ ] **(C)** It requires laying out a uniform $L^D$ mesh across all dimensions.
- [ ] **(D)** It can only be applied to 1D or 2D geometric problems.

In [5]:
ws.render_section('q02_pi_hit_probability')

**Q2 · easy** · 💡

In the hit-or-miss Monte Carlo estimation of $\pi$ on the unit quadrant $(x, y) \in [0, 1]^2$ (Slide 2), what is the exact theoretical probability $p$ that a single uniformly chosen dart lands inside the quarter-circle $x^2 + y^2 \le 1$ (to four decimal places)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\pi / 4 \approx 3.14159265 / 4$.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q03_pi_estimate_from_hits')

**Q3 · easy** · 💡

You run the quarter-circle dartboard code snippet from Slide 2 with $N = 2000$ random $(x, y)$ pairs in $[0, 1]^2$ and record `hits = 1573` points satisfying $x^2 + y^2 \le 1$. What is the resulting Monte Carlo estimate $\hat{\pi} = 4 \times \text{hits} / N$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $4 \times 1573 / 2000$.

</div>
</details>

_Numeric answer required._

In [7]:
ws.render_section('q04_pi_std_error_numeric')

**Q4 · medium** · 💡

What is the theoretical standard deviation (standard error) $\sigma_{\hat{\pi}} = \sqrt{\frac{\pi(4 - \pi)}{M}}$ of the quarter-circle $\hat{\pi}$ estimator when $M = 10{,}000$ independent darts are thrown (to four decimal places)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\sqrt{3.14159265 \times (4 - 3.14159265) / 10000} = \sqrt{2.69677} / 100$.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q05_extra_decimal_place_cost')

**Q5 · easy** · 💡

As highlighted on Slide 3 ("To get another decimal place takes 100 times longer!"), if a Monte Carlo simulation currently achieves a statistical error of $\varepsilon = 0.01$ in $T_{\text{CPU}} = 1\text{ minute}$, how many **minutes** of CPU time are required to reduce the statistical error by a factor of $10$ to $\varepsilon = 0.001$ (one additional decimal place)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $\varepsilon \propto M^{-1/2}$ and $T_{\text{CPU}} \propto M \propto \varepsilon^{-2}$, scaling $\varepsilon \to \varepsilon / 10$ scales $T_{\text{CPU}}$ by $10^2$.

</div>
</details>

_Numeric answer required._

In [9]:
ws.render_section('sec_curse_of_dimensionality')

## 2. The Curse of Dimensionality: Deterministic Quadrature vs. Monte Carlo Scaling

Why use a slow $M^{-1/2}$ method when deterministic 1D quadrature rules (like the Trapezoidal rule or Simpson's rule) converge much faster in 1D? The answer lies in the **Curse of Dimensionality** when integrating over a $D$-dimensional hypercube (Slides 3–5):
$$I = \int_0^1 dx_1 \cdots \int_0^1 dx_D\, f(x_1, \dots, x_D)$$

### Scaling of Conventional Deterministic Grid Quadrature
Suppose we lay out a uniform grid with $L$ points along each of the $D$ coordinate axes, so the grid spacing is $h = 1/L$:
- **Total number of grid points (and CPU time):**
  $$N = L^D = h^{-D} \sim T_{\text{CPU}} \quad \Longleftrightarrow \quad h = N^{-1/D}$$
- **Truncation error of an order-$\alpha$ quadrature rule:**
  - **Trapezoidal Rule ($\alpha = 2$):** From the Taylor expansion $f(x) = f(x_0) + f'(x_0)h + \frac{1}{2}f''(x_0)h^2 + \dots$, the leading integration error scales as $\varepsilon \sim f^{(2)}(x)\,h^2$.
  - **Simpson's Rule ($\alpha = 4$):** Error scales as $\varepsilon \sim f^{(4)}(x)\,h^4$.
  - **General Order-$\alpha$ Rule:**
    $$\varepsilon_{\text{quad}} \sim f^{(\alpha)}(x)\,h^\alpha = N^{-\alpha / D}$$
- **CPU Time as a function of target accuracy $\varepsilon$:** Inverting $\varepsilon \sim h^\alpha \implies h \sim \varepsilon^{1/\alpha}$ and substituting into $T_{\text{CPU}} \sim N = h^{-D}$ gives:
  $$T_{\text{quad}} \sim \varepsilon^{-D / \alpha}$$

### Scaling of Stochastic Integration (Monte Carlo)
By the Central Limit Theorem, averaging $M$ random evaluations of $f(\mathbf{x})$ yields a statistical uncertainty governed solely by the integrand variance $\sigma_f^2$:
$$\varepsilon_{\text{MC}} = \frac{\sigma_f}{\sqrt{M}} \sim M^{-1/2} \quad \Longrightarrow \quad T_{\text{MC}} \sim M \sim \varepsilon^{-2}$$
Crucially, the exponent $-2$ in $T_{\text{MC}} \sim \varepsilon^{-2}$ is **completely independent of the spatial dimension $D$!**

### The Exact Crossover Dimension $D^* = 2\alpha$
Comparing the CPU time exponents for small $\varepsilon \to 0$ (Slide 4):
$$\text{Monte Carlo beats an order-}\alpha\text{ grid rule whenever } \frac{D}{\alpha} > 2 \quad \Longleftrightarrow \quad \mathbf{D > 2\alpha}$$
- Against the **Trapezoidal Rule ($\alpha = 2$)**: MC wins for **$D > 4$** (and ties at $D = 4$).
- Against **Simpson's Rule ($\alpha = 4$)**: MC wins for **$D > 8$** (and ties at $D = 8$).
- In an atomistic simulation of just $100$ atoms in 3D, $D = 3N = 300 \gg 2\alpha$. Even a crude $L = 10$ grid per axis would require $N = 10^{300}$ function evaluations (vastly exceeding the $\sim 10^{80}$ atoms in the observable universe!), making Monte Carlo the **only** viable general integration method!

As Slide 5 summarizes, two further practical advantages of Monte Carlo are that it is **conceptually and practically simple** to code in any geometry, and it **comes with built-in statistical error bars**.


In [10]:
ws.render_section('concept_dimensionality_crossover')

### Interactive Explorer: Quadrature vs. Monte Carlo Scaling & Crossover Dimension D* = 2α

Select a deterministic quadrature rule (**Trapezoidal $\alpha=2$**, **Simpson's $\alpha=4$**, or **Gauss–Legendre $\alpha=6$**) and vary the **integration dimension $D$** (Slides 3–5). Toggle between **Error vs. Evaluations $\varepsilon(N)$** and **Log(CPU Time) vs. Dimension $D$** to see the exact crossover at $D^* = 2\alpha$.


**Key Takeaway:** On a log-log plot of CPU time vs. error (Slide 5), the Monte Carlo curve has a fixed slope of $-2$ regardless of $D$, whereas the grid quadrature slope $-D/\alpha$ steepens relentlessly with dimension, crossing MC at $D^* = 2\alpha$.


In [11]:
ws.render_section('q06_grid_points_hypercube')

**Q6 · easy** · 💡

You lay out a uniform deterministic grid with $L = 10$ points along each coordinate direction ($h = 1/L = 0.1$) to integrate a function in a $D = 6$-dimensional hypercube. What is the total number of grid points $N = L^D$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $10^6$.

</div>
</details>

_Numeric answer required._

In [12]:
ws.render_section('q07_trapezoidal_crossover_dim')

**Q7 · easy** · 💡

For the **Trapezoidal Rule** ($\varepsilon \sim h^2$, order $\alpha = 2$), above what exact spatial dimension $D^*$ does Monte Carlo integration ($\varepsilon \sim M^{-1/2}$) asymptotically beat deterministic grid quadrature in the limit of small error $\varepsilon \to 0$ (i.e., $D > D^*$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use the crossover formula $D^* = 2\alpha$ with $\alpha = 2$ (see Slide 3: 'MC is advantageous for D > 4!').

</div>
</details>

_Numeric answer required._

In [13]:
ws.render_section('q08_simpsons_crossover_dim')

**Q8 · easy** · 💡

For **Simpson's Rule** ($\varepsilon \sim f^{(4)}(x) h^4$, order $\alpha = 4$), above what crossover dimension $D^* = 2\alpha$ does Monte Carlo integration asymptotically require less CPU time than Simpson's rule as $\varepsilon \to 0$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $D^* = 2\alpha$ for $\alpha = 4$ (Slide 4).

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q09_quad_cpu_scaling_exponent')

**Q9 · medium** · 💡

Consider an order-$\alpha$ deterministic integration rule ($\varepsilon \sim h^\alpha$) applied to a $D$-dimensional integral using $N = L^D = h^{-D}$ grid points. How does the required CPU time $T_{\text{int}} \sim N$ scale with the target error $\varepsilon$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $\varepsilon \sim h^\alpha$, we have $h \sim \varepsilon^{1/\alpha}$. Substitute this into $T_{\text{int}} \sim h^{-D}$ (Slide 4).

</div>
</details>

- [ ] **(A)** $T_{\text{int}} \sim \varepsilon^{-D/\alpha}$
- [ ] **(B)** $T_{\text{int}} \sim \varepsilon^{-\alpha/D}$
- [ ] **(C)** $T_{\text{int}} \sim \varepsilon^{-2}$
- [ ] **(D)** $T_{\text{int}} \sim \varepsilon^{-D \alpha}$

In [15]:
ws.render_section('q10_trapezoidal_6d_error_exponent')

**Q10 · medium** · 💡

Suppose you use the **Trapezoidal Rule** ($\alpha = 2$) in $D = 6$ dimensions with $N$ total grid evaluations. The integration error scales with $N$ as $\varepsilon \propto N^{-\gamma}$. What is the numerical value of the positive exponent $\gamma = \alpha / D$ (to three decimal places)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $h = N^{-1/D}$ and $\varepsilon \propto h^\alpha$, the exponent is $\gamma = \alpha / D = 2 / 6 = 1/3$.

</div>
</details>

_Numeric answer required._

In [16]:
ws.render_section('q11_log_cpu_vs_log_error_plot')

**Q11 · medium** · 💡

On Slide 5, $\log(\text{CPU Time})$ is plotted against $\log(\text{error})$ (with smaller error to the left, labeled "Good", and larger error to the right, labeled "Bad") for Trapezoidal-rule quadrature in $2\text{D}, 4\text{D}, 6\text{D}, 8\text{D}$ alongside a band of orange lines labeled **MC**. Why is the $4\text{D}$ quadrature line parallel to the **MC** lines, while $6\text{D}$ and $8\text{D}$ are much steeper?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Take the logarithm of $T_{\text{quad}} \sim \varepsilon^{-D/2}$ and $T_{\text{MC}} \sim \varepsilon^{-2}$: what are the slopes $d(\log T)/d(\log \varepsilon)$?

</div>
</details>

- [ ] **(A)** Because $\log T_{\text{quad}} = -(D/2)\log\varepsilon + \text{const}$ has slope $-D/2$, which equals the Monte Carlo slope $-2$ at $D = 4$ and becomes steeper ($-3$ and $-4$) for $D = 6$ and $8$.
- [ ] **(B)** Because Monte Carlo only works in 4 dimensions.
- [ ] **(C)** Because the Trapezoidal rule is exact for all functions in $2\text{D}$.
- [ ] **(D)** Because random number generators run faster in $4\text{D}$ than in $2\text{D}$.

In [17]:
ws.render_section('q12_other_reasons_for_mc')

**Q12 · easy** · 💡

Besides its superior asymptotic scaling in high dimensions ($D > 2\alpha$), what additional practical advantages of Monte Carlo integration are highlighted on Slide 5? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the orange bullet points under 'Other reasons to do Monte Carlo:' on Slide 5.

</div>
</details>

- [ ] **(A)** It is conceptually and practically simple to implement, even for complex, high-dimensional, or irregularly shaped integration domains.
- [ ] **(B)** It comes with built-in statistical error bars (via the sample variance $\sigma / \sqrt{M}$).
- [ ] **(C)** It gives zero error with just $M = 1$ sample.
- [ ] **(D)** It requires knowing the analytical 4th derivative $f^{(4)}(x)$ of the integrand beforehand.

In [18]:
ws.render_section('sec_pdfs_moments_and_mappings')

### Probability Distribution Functions (PDFs), CDFs, and Moments (Slide 6)
A continuous random variable $x$ is governed by its **probability distribution function (p.d.f.)** $P(x)$, where $P(x)\,dx$ is the probability of observing a value in the infinitesimal interval $(x, x + dx)$:
$$P(x) \ge 0, \qquad \int_{-\infty}^{\infty} dx\, P(x) = 1 \quad \text{(Normalization)}$$

- **Cumulative Distribution Function (CDF):** The probability of observing a value less than or equal to $y$:
  $$C(y) = \int_{-\infty}^y dx\, P(x), \qquad 0 \le C(y) \le 1, \qquad \frac{dC(y)}{dy} = P(y)$$
- **Expectation Value (Average) of an Observable $g(x)$:**
  $$\langle g(x) \rangle = \bar{g} = \int_{-\infty}^{\infty} dx\, P(x)\, g(x)$$
- **Moments $I_n = \langle x^n \rangle = \int_{-\infty}^{\infty} dx\, P(x)\, x^n$:**
  - **Zeroth moment:** $I_0 = \langle x^0 \rangle = 1$ (normalization)
  - **First moment (Mean):** $\langle x \rangle = I_1$
  - **Variance:** $\sigma_x^2 = \left\langle (x - \langle x \rangle)^2 \right\rangle = \langle x^2 \rangle - \langle x \rangle^2 = I_2 - (I_1)^2$

### Mappings of Random Variables & Conservation of Probability (Slide 7)
Suppose we have a random variable $x$ with known distribution $p_x(x)$ (for instance, a uniform random number $x \sim \text{Uniform}(0, 1)$ where $p_x(x) = 1$) and we map it to a new variable $y = g(x)$. What is the probability density $p_y(y)$ of $y$?

Because the probability of landing in corresponding infinitesimal intervals must be conserved ($p_y(y)|dy| = p_x(x)|dx|$):
$$p_y(y) = p_x(x)\left|\frac{dx}{dy}\right| = p_x(x)\left|\frac{dx}{dg(x)}\right| = p_x(x)\left|\frac{dg(x)}{dx}\right|^{-1}$$

> [!NOTE] What happens when $y = g(x)$ is NOT monotonic? (Slide 7)
> If multiple distinct values $x^{(1)}, x^{(2)}, \dots$ map to the same output $y = g(x^{(k)})$, all those pre-image intervals contribute to $p_y(y)|dy|$, so we **sum over all branches**:
> $$p_y(y) = \sum_{k:\, g(x^{(k)}) = y} p_x\!\left(x^{(k)}\right) \left|\frac{dg}{dx}\Big|_{x^{(k)}}\right|^{-1}$$

**Worked Example (Slide 7):** Let $x \sim \text{Uniform}(0, 1]$ (so $p_x(x) = 1$) and define $y = g(x) = -\ln(x) \ge 0$.
Then $\frac{dg(x)}{dx} = -\frac{1}{x}$, so $\left|\frac{dg}{dx}\right|^{-1} = x = e^{-y}$. Therefore:
$$p_y(y)\,dy = (1)\cdot e^{-y}\,dy = e^{-y}\,dy \quad (y \ge 0)$$
More generally, scaling by $\lambda > 0$ via $y = -\frac{1}{\lambda}\ln(x)$ produces the **exponential (Poisson waiting-time) distribution** $p_y(y) = \lambda e^{-\lambda y}$!


In [19]:
ws.render_section('q13_pdf_normalization_numeric')

**Q13 · easy** · 💡

A continuous random variable $x \in [0, 2]$ has probability density function $P(x) = A x^2$ for $0 \le x \le 2$ (and $P(x) = 0$ otherwise). Using the normalization condition $I_0 = \int_{-\infty}^\infty P(x)\,dx = 1$, what is the numerical value of the normalization constant $A$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Integrate $\int_0^2 A x^2\,dx = A [x^3 / 3]_0^2 = \frac{8}{3}A = 1$, so $A = 3/8$.

</div>
</details>

_Numeric answer required._

In [20]:
ws.render_section('q14_variance_from_moments_numeric')

**Q14 · easy** · 💡

For a certain probability distribution $P(x)$, you compute the first moment $I_1 = \langle x \rangle = 3.0$ and the second moment $I_2 = \langle x^2 \rangle = 13.5$. Using the moment relation on Slide 6, what is the variance $\langle (x - \langle x \rangle)^2 \rangle = I_2 - (I_1)^2$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Subtract $(I_1)^2 = 3.0^2 = 9.0$ from $I_2 = 13.5$.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q15_cdf_properties')

**Q15 · easy** · 💡

Which of the following statements is **always true** for the cumulative distribution function $C(y) = \int_{-\infty}^y dx\,P(x)$ of any valid continuous probability density $P(x) \ge 0$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $P(x) \ge 0$ and $\int_{-\infty}^\infty P(x)\,dx = 1$, how does $C(y)$ behave as $y$ increases from $-\infty$ to $+\infty$?

</div>
</details>

- [ ] **(A)** $C(y)$ is monotonically non-decreasing with $0 \le C(y) \le 1$, $C(-\infty) = 0$, $C(+\infty) = 1$, and $\frac{dC(y)}{dy} = P(y)$.
- [ ] **(B)** $C(y)$ oscillates between $-1$ and $+1$ and integrates to $1$.
- [ ] **(C)** $C(y)$ equals the second derivative of $P(y)$.
- [ ] **(D)** $C(y)$ can decrease whenever $P(y)$ decreases.

In [22]:
ws.render_section('q16_mapping_jacobian_rule')

**Q16 · medium** · 💡

Let $x$ be a continuous random variable with PDF $p_x(x)$, and let $y = g(x)$ be a monotonic differentiable mapping. Which formula correctly gives the transformed PDF $p_y(y)$ (Slide 7)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Conservation of probability in corresponding intervals requires $p_y(y)|dy| = p_x(x)|dx|$.

</div>
</details>

- [ ] **(A)** $p_y(y) = p_x(x)\left|\frac{dx}{dy}\right| = p_x(x)\left|\frac{dg(x)}{dx}\right|^{-1}$
- [ ] **(B)** $p_y(y) = p_x(x)\left|\frac{dg(x)}{dx}\right|$
- [ ] **(C)** $p_y(y) = p_x(g(x))$
- [ ] **(D)** $p_y(y) = \frac{d}{dy} p_x(x)$

In [23]:
ws.render_section('q17_power_mapping_pdf_numeric')

**Q17 · medium** · 💡

Suppose $x \sim \text{Uniform}(0, 1)$ (so $p_x(x) = 1$ on $0 < x < 1$) and we define a new random variable $y = g(x) = x^2$. Using $p_y(y) = p_x(x)\left|\frac{dg}{dx}\right|^{-1} = \frac{1}{2\sqrt{y}}$ for $0 < y < 1$, what is the numerical value of the probability density $p_y(y)$ evaluated at $y = 0.25$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $p_y(0.25) = 1 / (2\sqrt{0.25}) = 1 / (2 \times 0.5)$.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q18_non_monotonic_mapping')

**Q18 · hard** · 💡

Slide 7 asks: *"What happens when $g(x)$ is not monotonic?"* Suppose $x$ is distributed uniformly on $[-1, 1]$ with $p_x(x) = \frac{1}{2}$, and we define the non-monotonic mapping $y = g(x) = x^2 \in (0, 1]$. Because both $x_+ = +\sqrt{y}$ and $x_- = -\sqrt{y}$ map to the same $y$, what is the resulting PDF $p_y(y)$ on $0 < y \le 1$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum the Jacobian contributions $\sum_{x = \pm\sqrt{y}} p_x(x) |2x|^{-1}$ over both branches!

</div>
</details>

- [ ] **(A)** $p_y(y) = \frac{1}{2\sqrt{y}}$, because the two branches $x = +\sqrt{y}$ and $x = -\sqrt{y}$ each contribute $\frac{1}{2} \cdot \frac{1}{2\sqrt{y}} = \frac{1}{4\sqrt{y}}$.
- [ ] **(B)** $p_y(y) = 0$, because the positive and negative branches cancel out.
- [ ] **(C)** $p_y(y) = \frac{1}{4\sqrt{y}}$, because only the positive branch counts.
- [ ] **(D)** $p_y(y) = 2\sqrt{y}$.

In [25]:
ws.render_section('sec_inverse_transform_and_box_muller')

### Inverse Transform Sampling: $y = C^{-1}(x)$ (Slides 8–9)
How do we turn a standard uniform random number generator $x \sim \text{Uniform}(0, 1)$ (where $p_x(x) = 1$ on $0 \le x \le 1$) into a generator for **any** desired target distribution $p_y(y) = f(y)$?

Set the mapping relation $\frac{dx}{dy} = f(y)$ and integrate:
$$x = C(y) = \int_{-\infty}^y du\, f(u) \quad \Longleftrightarrow \quad y(x) = C^{-1}(x)$$
**Geometric & Physical Interpretation (Slide 9):**
1. Draw a uniform random deviate $x \in (0, 1]$.
2. Find the value $y$ on the horizontal axis that has **fraction $x$ of the total area under $f(y)$ to the left of $y$**, i.e., solve $C(y) = x$.
3. Return $y = C^{-1}(x)$.

#### Key Analytical Examples of Inverse Transform Sampling
1. **Exponential / Poisson Waiting-Time Distribution (Slide 9):**
   - Target PDF: $f(y) = \lambda e^{-\lambda y}$ for $y \ge 0$.
   - CDF: $x = C(y) = \int_0^y \lambda e^{-\lambda u}\,du = 1 - e^{-\lambda y}$.
   - Inversion: $1 - x = e^{-\lambda y}$. Since $x' = 1 - x$ is also uniformly distributed on $(0, 1]$, we obtain:
     $$y = -\frac{1}{\lambda}\ln(1 - x) \;\stackrel{d}{=}\; -\frac{1}{\lambda}\ln(x')$$
2. **Power-Law Distribution on $[0, 1]$:**
   - Target PDF: $f(y) = (n + 1)y^n$ for $0 \le y \le 1$ ($n > -1$).
   - CDF: $x = C(y) = y^{n+1} \implies y = x^{1/(n+1)}$.
3. **Cauchy / Lorentzian Distribution:**
   - Target PDF: $f(y) = \frac{1}{\pi}\frac{\gamma}{y^2 + \gamma^2}$ for $-\infty < y < \infty$.
   - CDF: $x = C(y) = \frac{1}{2} + \frac{1}{\pi}\arctan(y/\gamma) \implies y = \gamma \tan\!\left[\pi\left(x - \frac{1}{2}\right)\right]$.

### Sampling a Normal Gaussian: The Box–Muller Transform (Slide 10)
In Molecular Dynamics (Lesson 11!), initializing Maxwell–Boltzmann velocities or running an Andersen/Langevin thermostat requires drawing Gaussian random numbers:
$$p(y)\,dy = \frac{1}{\sqrt{2\pi}}e^{-y^2/2}\,dy$$
Why can't we just use 1D Inverse Transform Sampling? Because the Gaussian indefinite integral $C(y) = \Phi(y) = \frac{1}{2}\left[1 + \text{erf}(y/\sqrt{2})\right]$ **cannot be inverted in closed form using elementary functions!**

**Box and Muller (1958)** solved this ingeniously by pairing **two** independent uniform deviates $x_1, x_2 \in (0, 1)$ to generate **two** independent standard normal deviates $y_1, y_2$ via 2D polar coordinates:
$$p(y_1, y_2)\,dy_1\,dy_2 = \frac{1}{2\pi}e^{-(y_1^2 + y_2^2)/2}\,dy_1\,dy_2 = p(x_1, x_2)\left|\frac{\partial(x_1, x_2)}{\partial(y_1, y_2)}\right| dy_1\,dy_2$$
Define the forward and inverse coordinate transformations (Slide 10):
$$y_1 = \sqrt{-2\ln x_1}\cos(2\pi x_2), \qquad y_2 = \sqrt{-2\ln x_1}\sin(2\pi x_2)$$
$$x_1 = \exp\!\left(-\frac{1}{2}[y_1^2 + y_2^2]\right), \qquad x_2 = \frac{1}{2\pi}\tan^{-1}\!\left(\frac{y_2}{y_1}\right)$$
The $2\times 2$ Jacobian determinant factors cleanly into the product of two 1D Gaussians:
$$\frac{\partial(x_1, x_2)}{\partial(y_1, y_2)} = -\left[\frac{1}{\sqrt{2\pi}}e^{-y_1^2/2}\right]\left[\frac{1}{\sqrt{2\pi}}e^{-y_2^2/2}\right]$$
proving that $y_1$ and $y_2$ are **statistically independent** standard normal variables! (To obtain mean $\mu$ and standard deviation $\sigma = \sqrt{k_B T / m}$, simply set $v = \mu + \sigma y$).

### The Marsaglia Polar Method: Avoiding $\sin$ and $\cos$ (Slide 10 Bottom)
Evaluating transcendental functions $\cos(2\pi x_2)$ and $\sin(2\pi x_2)$ is computationally expensive. At the bottom of Slide 10 ("Better:"), we instead pick a random point $(v_1, v_2)$ uniformly inside the square $[-1, 1]^2$ and accept it if it lies inside the unit disk:
$$R^2 = v_1^2 + v_2^2 \le 1 \quad (\text{and } R^2 > 0)$$
Because $R^2$ is uniformly distributed on $(0, 1]$ and the angle $\theta$ is uniform on $[0, 2\pi)$, we can identify $x_1 = R^2$ (or $\sqrt{R^2}$ in radial parameterization) and replace $\cos(2\pi x_2)$ and $\sin(2\pi x_2)$ with the algebraic ratios $v_1/\sqrt{R^2}$ and $v_2/\sqrt{R^2}$:
$$y_1 = \sqrt{-2\ln(R^2)}\,\frac{v_1}{\sqrt{R^2}}, \qquad y_2 = \sqrt{-2\ln(R^2)}\,\frac{v_2}{\sqrt{R^2}}$$
- **No $\sin$ or $\cos$ evaluation required!**
- **Two independent Gaussian deviates per calculation** ("1 for now, 2 for next time").


In [26]:
ws.render_section('concept_inverse_and_box_muller')

### Interactive Explorer: Inverse Transform Sampling & 2D Box–Muller Gaussian Mapping

Switch between **Exponential ($y = -\ln(1-u)/\lambda$)**, **Cauchy/Lorentzian ($y = \gamma\tan(\pi(u-1/2))$)**, and the **2D Box–Muller Transform** (Slides 8–10). Observe how quantile inversion $u \to C^{-1}(u)$ maps uniform draws on $(0,1)$ into non-uniform densities, and how Box–Muller maps $(x_1, x_2)$ into concentric 2D Gaussian rings.


**Key Takeaway:** 1D Inverse Transform Sampling works whenever the indefinite integral $C(y) = \int_{-\infty}^y f(u)\,du$ is analytically known and invertible. When $C^{-1}$ has no closed form—as for the Gaussian—2D coordinate transformations like **Box–Muller** make exact analytical inversion possible!


In [27]:
ws.render_section('q19_inverse_transform_requirements')

**Q19 · easy** · 💡

According to Slide 8 ("What is the Mapping Doing?"), what mathematical condition must the target probability density $p(y)$ satisfy in order to directly apply 1D **Inverse Transform Sampling** $y = C^{-1}(x)$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the second bullet point in the yellow box at the bottom of Slide 8.

</div>
</details>

- [ ] **(A)** The definite/indefinite integral $C(y) = \int_{-\infty}^y du\,p(u)$ must be known and analytically (or efficiently numerically) invertible.
- [ ] **(B)** The target distribution $p(y)$ must be symmetric about $y = 0$.
- [ ] **(C)** The target distribution $p(y)$ must be a constant function.
- [ ] **(D)** The second derivative $p''(y)$ must be zero everywhere.

In [28]:
ws.render_section('q20_inverse_transform_quantile_meaning')

**Q20 · easy** · 💡

In 1D Inverse Transform Sampling, suppose the uniform random number generator returns $x = 0.25$. Geometrically, how is the output deviate $y = C^{-1}(0.25)$ related to the target probability density curve $p(y)$ (Slide 9)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $C(y) = \int_{-\infty}^y p(u)\,du$ is the area under $p(y)$ to the left of $y$, what does $C(y) = 0.25$ mean?

</div>
</details>

- [ ] **(A)** $y$ is the value (first quartile) such that exactly fraction $x = 0.25$ ($25\%$) of the total area under $p(y)$ lies to the left of $y$.
- [ ] **(B)** $y$ is the point where the height of the PDF equals $p(y) = 0.25$.
- [ ] **(C)** $y$ is always equal to $0.25$ times the mean $\langle y \rangle$.
- [ ] **(D)** $y$ is the point where the slope $dp/dy = 0.25$.

In [29]:
ws.render_section('q21_exponential_sampling_formula')

**Q21 · easy** · 💡

Given a uniform deviate $x \in (0, 1]$, which formula generates a random waiting time $y \ge 0$ drawn from the exponential (Poisson event) distribution $p(y) = \lambda e^{-\lambda y}$ (Slide 9)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Invert $x = C(y) = 1 - e^{-\lambda y}$ and note that $x' = 1 - x$ is also uniform on $(0, 1]$.

</div>
</details>

- [ ] **(A)** $y = -\frac{1}{\lambda}\ln(x)$ (or equivalently $-\frac{1}{\lambda}\ln(1 - x)$)
- [ ] **(B)** $y = \lambda e^{-\lambda x}$
- [ ] **(C)** $y = \sqrt{-2\lambda \ln x}$
- [ ] **(D)** $y = -\lambda \ln(x)$

In [30]:
ws.render_section('q22_exponential_deviate_numeric')

**Q22 · easy** · 💡

Using the exponential sampling formula $y = -\frac{\ln(x)}{\lambda}$ from Slide 9 with decay rate $\lambda = 2.0\text{ ps}^{-1}$, what waiting time $y$ (in $\text{ps}$, to three decimal places) is generated when the uniform deviate is $x = e^{-3} \approx 0.049787$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $\ln(x) = \ln(e^{-3}) = -3$ and $\lambda = 2.0$ into $y = -\ln(x)/\lambda$.

</div>
</details>

_Numeric answer required._

In [31]:
ws.render_section('q23_exponential_complement_trick')

**Q23 · medium** · 💡

When inverting the exponential CDF $x = 1 - e^{-\lambda y}$, algebra gives $y = -\frac{1}{\lambda}\ln(1 - x)$, yet codes almost always write $y = -\frac{1}{\lambda}\ln(x)$ directly (Slide 9). Why is replacing $1 - x$ with $x$ mathematically valid, and why must $x$ be drawn from $(0, 1]$ rather than $[0, 1)$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If $x \sim \text{Uniform}(0, 1)$, what is the distribution of $x' = 1 - x$? And what happens to $\ln(x)$ if $x = 0$?

</div>
</details>

- [ ] **(A)** If $x$ is uniform on $(0, 1)$, then $x' = 1 - x$ has the exact same $\text{Uniform}(0, 1)$ distribution; furthermore, $x = 0$ must be excluded so $\ln(x)$ does not diverge to $-\infty$.
- [ ] **(B)** Because $\ln(1 - x) = \ln(x)$ for all real numbers $x$.
- [ ] **(C)** Because $1 - x$ is only approximately uniform when $\lambda \gg 1$.
- [ ] **(D)** Replacing $1 - x$ with $x$ changes the mean of $y$ from $1/\lambda$ to $2/\lambda$.

In [32]:
ws.render_section('q24_power_law_inverse_cdf')

**Q24 · medium** · 💡

You want to sample a continuous random variable $y \in [0, 1]$ from the cubic probability density $p(y) = 4y^3$ using a uniform random deviate $u \sim \text{Uniform}(0, 1)$. What is the Inverse Transform Sampling formula $y(u) = C^{-1}(u)$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $C(y) = \int_0^y 4t^3\,dt = y^4$. Then set $C(y) = u$ and solve for $y$.

</div>
</details>

- [ ] **(A)** $y = u^{1/4}$
- [ ] **(B)** $y = u^4$
- [ ] **(C)** $y = \frac{1}{4}u^{1/3}$
- [ ] **(D)** $y = -\frac{1}{4}\ln u$

In [33]:
ws.render_section('q25_power_law_inverse_numeric')

**Q25 · medium** · 💡

Using the inverse transform $y = u^{1/3}$ for the quadratic PDF $p(y) = 3y^2$ on $[0, 1]$, what value of $y$ is returned when the uniform random number is $u = 0.125$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Take the cube root of $0.125 = 1/8$.

</div>
</details>

_Numeric answer required._

In [34]:
ws.render_section('q26_linear_pdf_inverse_numeric')

**Q26 · medium** · 💡

Suppose a random variable $y \in [0, 2]$ has the normalized linear PDF $p(y) = \frac{1}{2}y$ for $0 \le y \le 2$. Find the cumulative distribution function $C(y) = \int_0^y \frac{1}{2}t\,dt$, invert $C(y) = u$, and compute the sampled deviate $y$ when $u = 0.36$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $C(y) = \frac{1}{4}y^2 = u$, we have $y = 2\sqrt{u}$.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q27_cauchy_inverse_transform')

**Q27 · medium** · 💡

The standard **Cauchy (Lorentzian)** distribution $p(y) = \frac{1}{\pi(1 + y^2)}$ on $-\infty < y < \infty$ has cumulative distribution function $C(y) = \frac{1}{2} + \frac{1}{\pi}\arctan(y)$. If $u \sim \text{Uniform}(0, 1)$, what is the exact Inverse Transform Sampling formula $y(u) = C^{-1}(u)$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $u = \frac{1}{2} + \frac{1}{\pi}\arctan(y)$ for $y$.

</div>
</details>

- [ ] **(A)** $y = \tan\!\left[\pi\left(u - \frac{1}{2}\right)\right]$
- [ ] **(B)** $y = \arctan(\pi u)$
- [ ] **(C)** $y = \frac{1}{\pi(1 + u^2)}$
- [ ] **(D)** $y = \sin(2\pi u)$

In [36]:
ws.render_section('q28_cauchy_numeric')

**Q28 · medium** · 💡

Using the Cauchy inverse transform $y = \gamma \tan\!\left[\pi\left(u - \frac{1}{2}\right)\right]$ with width parameter $\gamma = 2.0$, what deviate $y$ is generated when $u = 0.75$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Note that $\pi(0.75 - 0.50) = \pi/4$, and $\tan(\pi/4) = 1$.

</div>
</details>

_Numeric answer required._

In [37]:
ws.render_section('q29_why_gaussian_needs_box_muller')

**Q29 · easy** · 💡

Why can we **not** use simple 1D analytical Inverse Transform Sampling $y = C^{-1}(x)$ to draw Normal Gaussian random numbers $p(y) = \frac{1}{\sqrt{2\pi}}e^{-y^2/2}$, motivating the 2D **Box–Muller** method instead?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Can the 1D Gaussian integral $\Phi(y) = \int_{-\infty}^y \frac{1}{\sqrt{2\pi}}e^{-u^2/2}\,du$ be expressed and inverted in terms of elementary functions?

</div>
</details>

- [ ] **(A)** Because the 1D Gaussian cumulative distribution function $\Phi(y)$ (the error function $\text{erf}$) cannot be written or inverted in closed form using elementary functions, whereas the 2D Gaussian integral in polar coordinates $r\,dr\,d\theta\,e^{-r^2/2}$ integrates and inverts analytically.
- [ ] **(B)** Because the Gaussian PDF $p(y)$ can be negative for $y < 0$.
- [ ] **(C)** Because the 1D Gaussian integral $\int_{-\infty}^\infty e^{-y^2/2}\,dy$ diverges to infinity.
- [ ] **(D)** Because 1D Inverse Transform Sampling only works for discrete variables.

In [38]:
ws.render_section('q30_box_muller_formulas')

**Q30 · easy** · 💡

Given two independent uniform deviates $x_1, x_2 \in (0, 1)$, what are the **Box–Muller transformation formulas** (Slide 10) that produce two independent standard normal deviates $y_1, y_2 \sim \mathcal{N}(0, 1)$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the orange equations in the middle of Slide 10.

</div>
</details>

- [ ] **(A)** $y_1 = \sqrt{-2\ln x_1}\cos(2\pi x_2)$ and $y_2 = \sqrt{-2\ln x_1}\sin(2\pi x_2)$
- [ ] **(B)** $y_1 = -\ln(x_1)\cos(x_2)$ and $y_2 = -\ln(x_1)\sin(x_2)$
- [ ] **(C)** $y_1 = x_1 \cos(2\pi x_2)$ and $y_2 = x_1 \sin(2\pi x_2)$
- [ ] **(D)** $y_1 = \sqrt{x_1 + x_2}$ and $y_2 = \sqrt{x_1 - x_2}$

In [39]:
ws.render_section('q31_box_muller_numeric')

**Q31 · medium** · 💡

In the Box–Muller algorithm, suppose your uniform random number generator produces the pair $x_1 = e^{-2} \approx 0.135335$ and $x_2 = \frac{1}{6} \approx 0.166667$. Compute the first standard normal deviate $y_1 = \sqrt{-2\ln x_1}\cos(2\pi x_2)$.


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First evaluate $r = \sqrt{-2\ln(e^{-2})} = \sqrt{4} = 2$, and $\cos(2\pi / 6) = \cos(\pi / 3) = 0.5$.

</div>
</details>

_Numeric answer required._

In [40]:
ws.render_section('q32_box_muller_inverse_x1_numeric')

**Q32 · medium** · 💡

Using the inverse Box–Muller relation on Slide 10,
$$x_1 = \exp\!\left(-\frac{1}{2}\left[y_1^2 + y_2^2\right]\right)$$
if a generated Gaussian pair happens to fall on the circle of radius $\sqrt{y_1^2 + y_2^2} = \sqrt{2\ln 4}$ (so $y_1^2 + y_2^2 = 2\ln 4$), what was the value of the input uniform deviate $x_1$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $y_1^2 + y_2^2 = 2\ln 4$ into $\exp(-\frac{1}{2}[y_1^2 + y_2^2]) = \exp(-\ln 4)$.

</div>
</details>

_Numeric answer required._

In [41]:
ws.render_section('q33_box_muller_jacobian_factorization')

**Q33 · medium** · 💡

On Slide 10, the $2\times 2$ Jacobian determinant of the Box–Muller transformation is shown to be:
$$\frac{\partial(x_1, x_2)}{\partial(y_1, y_2)} = -\left[\frac{1}{\sqrt{2\pi}}e^{-y_1^2/2}\right]\left[\frac{1}{\sqrt{2\pi}}e^{-y_2^2/2}\right]$$
Why does the fact that the joint density $p(y_1, y_2) = \left|\frac{\partial(x_1, x_2)}{\partial(y_1, y_2)}\right|$ factors into the product $p(y_1)\,p(y_2)$ have such an important practical consequence?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In probability theory, what does $p(y_1, y_2) = p(y_1)p(y_2)$ prove about the random variables $y_1$ and $y_2$?

</div>
</details>

- [ ] **(A)** It proves that $y_1$ and $y_2$ are statistically independent Normal Gaussian variables, so both can be used as independent draws in a simulation.
- [ ] **(B)** It proves that $y_1$ and $y_2$ are always equal to each other ($y_1 = y_2$).
- [ ] **(C)** It proves that $y_1 + y_2 = 0$.
- [ ] **(D)** It shows that one of the two variables must be discarded.

In [42]:
ws.render_section('q34_marsaglia_polar_advantage')

**Q34 · easy** · 💡

At the bottom of Slide 10 ("Better: Pick $R^2 = v_1^2 + v_2^2$"), what is the primary computational advantage of the **Marsaglia polar rejection variant** of Box–Muller?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the blue box at the bottom of Slide 10: what expensive functions are eliminated by using $v_1/\sqrt{R^2}$ and $v_2/\sqrt{R^2}$?

</div>
</details>

- [ ] **(A)** It avoids evaluating slow trigonometric functions ($\sin$ and $\cos$) by replacing $\cos(2\pi x_2)$ and $\sin(2\pi x_2)$ with the algebraic ratios $v_1/\sqrt{R^2}$ and $v_2/\sqrt{R^2}$, while still yielding two independent Gaussian numbers per accepted pair.
- [ ] **(B)** It eliminates the square root and natural logarithm operations.
- [ ] **(C)** It only requires 1 uniform random number to generate 10 Gaussian numbers.
- [ ] **(D)** It works without a random number generator.

In [43]:
ws.render_section('q35_marsaglia_acceptance_rate')

**Q35 · medium** · 💡

In the Marsaglia polar variant of Box–Muller, candidate pairs $(v_1, v_2)$ are drawn uniformly from the square $[-1, 1] \times [-1, 1]$ (area $4$) and accepted only if they fall inside the unit disk $R^2 = v_1^2 + v_2^2 \le 1$ (area $\pi$). What is the exact acceptance probability $P_{\text{accept}}$ of a candidate pair (to four decimal places)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice the connection back to Slide 2: ratio of circle area $\pi$ to square area $4$!

</div>
</details>

_Numeric answer required._

In [44]:
ws.render_section('q36_marsaglia_polar_numeric')

**Q36 · medium** · 💡

In the Marsaglia polar method, suppose you draw $(v_1, v_2) = (0.30, 0.40)$ inside the unit circle, so $R = \sqrt{v_1^2 + v_2^2} = 0.50$ and $R^2 = 0.25$. Using $y_1 = \sqrt{-2\ln(R^2)} \frac{v_1}{\sqrt{R^2}}$, compute the resulting standard normal deviate $y_1$ (to three decimal places).


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\sqrt{-2\ln(0.25)} \times (0.30 / 0.50) = \sqrt{2.7725887} \times 0.60$.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q37_maxwell_boltzmann_velocity_scaling')

**Q37 · medium** · 💡

Connecting Lesson 12 back to Lesson 11 (Slide 2 of `08.ensemble.pdf`): to initialize an MD simulation or perform an Andersen thermostat kick at temperature $T$, each velocity component must follow the Maxwell–Boltzmann Gaussian distribution
$$P(v_x) \propto \exp\!\left(-\frac{m v_x^2}{2 k_B T}\right)$$
which is a Normal distribution with mean $0$ and standard deviation $\sigma_v = \sqrt{k_B T / m}$. If Box–Muller returns a standard normal deviate $y_1 = -1.50$ for an atom of mass $m = 4.0$ at temperature $k_B T = 9.0$ (in reduced units), what is the assigned velocity $v_x = \sigma_v y_1$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute $\sigma_v = \sqrt{9.0 / 4.0} = 1.50$, then multiply by $y_1 = -1.50$.

</div>
</details>

_Numeric answer required._

In [46]:
ws.render_section('q38_comprehensive_sampling_match')

**Q38 · medium** · 💡

Which of the following statements connecting random-variable mappings, Monte Carlo integration, and atomistic simulation are **correct**? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Verify each statement against the formulas on Slides 4, 7, 9, and 10.

</div>
</details>

- [ ] **(A)** To sample the time $\Delta t_{\text{wait}}$ until the next stochastic heat-bath collision in a Poisson process of rate $\nu$, one uses Inverse Transform Sampling $y = -\ln(u)/\nu$.
- [ ] **(B)** To sample 3D Maxwell–Boltzmann velocities $(v_x, v_y, v_z)$ for $N$ atoms, one can generate standard normal pairs $(y_1, y_2)$ via Box–Muller and multiply by $\sqrt{k_B T / m}$.
- [ ] **(C)** For evaluating the 300-dimensional configurational integral of 100 atoms in 3D, Simpson's rule ($\alpha = 4$) is asymptotically faster than Monte Carlo because $\alpha = 4 > 2$.
- [ ] **(D)** When transforming random variables in $D$ dimensions, probability conservation requires multiplying by the absolute value of the Jacobian determinant $|\det(\partial \mathbf{x} / \partial \mathbf{y})|$.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()